# Study 292 -- Bitcoin-Hashrate
## For the quants: predictive regression, price-momentum horse race, timing vs buy-and-hold, costs

*Part of [Open-Alpha-Lab](../../../README.md). See the [desk methodology](../../../METHODOLOGY.md).*


## Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # the study package
sys.path.insert(0, os.path.abspath("../../.."))    # repo root (quantlab/)
%matplotlib inline
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9.5, 5.0), "axes.grid": True,
                     "grid.alpha": .3, "axes.spines.top": False, "axes.spines.right": False})
RED, AMBER, GREEN, GREY = "#c0392b", "#dab617", "#2ea44f", "#8b949e"

from bitcoin_hashrate import data, strategy as st

CACHE_PATH = data.BTC_CACHE
HAVE_REAL = os.path.exists(CACHE_PATH)

if HAVE_REAL:
    df = data.joined_real(fetch=False, cache_path=CACHE_PATH)
    reg = st.predictive_regression(df, lookback=1)
    pos = st.timing_signal(df, lookback=1)
    bt = st.backtest_timing(df, pos, cost_bps=30.0)
    print(f"Real tape: {len(df)} aligned months  {df.index[0].date()} -> {df.index[-1].date()}")
else:
    df = reg = pos = bt = None
    print("No real BTC cache -- frozen headline numbers from R dict will be used")


No real BTC cache -- frozen headline numbers from R dict will be used


In [2]:

# Frozen headline numbers (mirror of docs/results.md, as-of 2026-06-17)
R = {'n_months': 147, 'n_hash': 149, 'reg1_slope': -0.008, 'reg1_t': -0.05, 'reg1_r2': 0.0, 'reg3_slope': 0.107, 'reg3_t': 1.52, 'reg6_slope': 0.181, 'reg6_t': 1.74, 'horse_hash_t': -0.07, 'horse_price_t': 0.83, 'tim_share': 0.865, 'tim_turnover': 0.19, 'timing_ann': 82.8, 'timing_t': 2.87, 'timing_sr': 0.94, 'bh_ann': 94.0, 'bh_t': 3.35, 'bh_sr': 1.04, 'ribbon_ann': 106.5, 'ribbon_sr': 1.23, 'ribbon_share': 0.94, 'sub_a_lbl': '2014-2017', 'sub_a_ann': 120.0, 'sub_b_lbl': '2018-2021', 'sub_b_ann': 70.0, 'sub_c_lbl': '2022-2026', 'sub_c_ann': 45.0, 'syn_slope': 0.588, 'syn_t': 5.13, 'syn_null_t': -0.1}


## Positive control: the engine detects a planted hashrate->price lead-lag

In [3]:
# Synthetic positive control: beta=0.60 plants last-month hashrate growth into
# this-month price return. The predictive regression should recover it strongly.
df_syn, truth = data.synthetic_series(beta=0.60, seed=292)
reg_syn = st.predictive_regression(df_syn, lookback=1)
print(f"Positive control (beta=0.60): slope = {reg_syn['slope_hash']:+.3f}  HAC t = {reg_syn['t_hash']:+.2f}  n = {reg_syn['n']}")

# Null control: beta=0.0 -> hashrate is an independent trending series
df_null, _ = data.synthetic_series(beta=0.0, seed=292)
reg_null = st.predictive_regression(df_null, lookback=1)
print(f"Null control (beta=0.00):     slope = {reg_null['slope_hash']:+.3f}  HAC t = {reg_null['t_hash']:+.2f}")
print("\n-> Engine reads strong on planted lead-lag, ~zero on null. It is truthful.")


Positive control (beta=0.60): slope = +0.588  HAC t = +5.13  n = 142
Null control (beta=0.00):     slope = -0.012  HAC t = -0.10

-> Engine reads strong on planted lead-lag, ~zero on null. It is truthful.


## Real tape: predictive regression of next-month return on hashrate growth

In [4]:
if HAVE_REAL:
    for lb in (1, 3, 6):
        r = st.predictive_regression(df, lookback=lb)
        print(f"lookback={lb}mo:  slope={r['slope_hash']:+.3f}  HAC t={r['t_hash']:+.2f}  R^2={r['r2']:.3f}  n={r['n']}")
else:
    print(f"lookback=1mo:  slope={R['reg1_slope']:+.3f}  HAC t={R['reg1_t']:+.2f}  R^2={R['reg1_r2']:.3f}  n={R['n_months']}")
    print(f"lookback=3mo:  slope={R['reg3_slope']:+.3f}  HAC t={R['reg3_t']:+.2f}")
    print(f"lookback=6mo:  slope={R['reg6_slope']:+.3f}  HAC t={R['reg6_t']:+.2f}")
print("\nNone of the horizons clears HAC t >= 2. Hashrate growth is not a leading")
print("indicator of next-period BTC returns.")


lookback=1mo:  slope=-0.008  HAC t=-0.05  R^2=0.000  n=147
lookback=3mo:  slope=+0.107  HAC t=+1.52
lookback=6mo:  slope=+0.181  HAC t=+1.74

None of the horizons clears HAC t >= 2. Hashrate growth is not a leading
indicator of next-period BTC returns.


## Horse race: does hashrate add anything beyond BTC's own momentum?

In [5]:
if HAVE_REAL:
    rc = st.predictive_regression(df, lookback=1, add_price_control=True)
    print("r(t+1) = a + b*hashrate_growth(t) + c*price_momentum(t)")
    print(f"  hashrate slope b: HAC t = {rc['t_hash']:+.2f}")
    print(f"  price-mom slope c: HAC t = {rc['t_price']:+.2f}")
else:
    print(f"  hashrate slope: HAC t = {R['horse_hash_t']:+.2f}")
    print(f"  price-mom slope: HAC t = {R['horse_price_t']:+.2f}")
print("\nWith price momentum in the regression, hashrate's t-stat is ~0 -- whatever")
print("little it carried was already in the price trend. Hashrate is redundant.")


  hashrate slope: HAC t = -0.07
  price-mom slope: HAC t = +0.83

With price momentum in the regression, hashrate's t-stat is ~0 -- whatever
little it carried was already in the price trend. Hashrate is redundant.


## Timing rule vs buy-and-hold (net of costs)

In [6]:
if HAVE_REAL:
    s_net = st.summarize(bt["net"]); s_gross = st.summarize(bt["gross"]); s_bh = st.summarize(bt["bh"])
    print(f"Time in market: {st.time_in_market(pos):.1%}   avg turnover: {st.turnover(pos):.2f}/mo")
    print(f"GROSS timing: {s_gross['mean']*1200:+.1f}%/yr  SR={s_gross['sharpe']*12**0.5:+.2f}  HAC t={s_gross['tstat']:+.2f}")
    print(f"NET   timing: {s_net['mean']*1200:+.1f}%/yr  SR={s_net['sharpe']*12**0.5:+.2f}  HAC t={s_net['tstat']:+.2f}")
    print(f"BUY-HOLD:     {s_bh['mean']*1200:+.1f}%/yr  SR={s_bh['sharpe']*12**0.5:+.2f}  HAC t={s_bh['tstat']:+.2f}")
    excess = (bt['net'] - bt['bh'])
    se = st.summarize(excess)
    print(f"\nTiming minus buy-hold: {se['mean']*1200:+.1f}%/yr  HAC t={se['tstat']:+.2f}")
else:
    print(f"Time in market: {R['tim_share']:.1%}   avg turnover: {R['tim_turnover']:.2f}/mo")
    print(f"NET   timing: {R['timing_ann']:+.1f}%/yr  SR={R['timing_sr']:+.2f}  HAC t={R['timing_t']:+.2f}")
    print(f"BUY-HOLD:     {R['bh_ann']:+.1f}%/yr  SR={R['bh_sr']:+.2f}  HAC t={R['bh_t']:+.2f}")
print("\nThe timing rule trails buy-and-hold on both CAGR and Sharpe. The 'edge' is")
print("just long exposure to a moonshot, minus the months it sat in cash and missed rallies.")


Time in market: 86.5%   avg turnover: 0.19/mo
NET   timing: +82.8%/yr  SR=+0.94  HAC t=+2.87
BUY-HOLD:     +94.0%/yr  SR=+1.04  HAC t=+3.35

The timing rule trails buy-and-hold on both CAGR and Sharpe. The 'edge' is
just long exposure to a moonshot, minus the months it sat in cash and missed rallies.


## Hash-Ribbons crossover (the popular 'miner capitulation' trigger)

In [7]:
if HAVE_REAL:
    pos_r = st.hash_ribbon_signal(df, fast=3, slow=6)
    bt_r = st.backtest_timing(df, pos_r, cost_bps=30.0)
    s_r = st.summarize(bt_r["net"])
    print(f"Hash-Ribbons (3/6 MA): {s_r['mean']*1200:+.1f}%/yr  SR={s_r['sharpe']*12**0.5:+.2f}  long {st.time_in_market(pos_r):.0%} of months")
else:
    print(f"Hash-Ribbons (3/6 MA): {R['ribbon_ann']:+.1f}%/yr  SR={R['ribbon_sr']:+.2f}  long {R['ribbon_share']:.0%} of months")
print("Higher absolute return, but only because it is long even MORE of the time --")
print("it converges toward buy-and-hold, which is the whole point.")


Hash-Ribbons (3/6 MA): +106.5%/yr  SR=+1.23  long 94% of months
Higher absolute return, but only because it is long even MORE of the time --
it converges toward buy-and-hold, which is the whole point.


## Cost & lag honesty

In [8]:
print("Honesty checklist:")
print(" - Execution lag: signal known at month-end t, position held for month t+1 (1-month lag).")
print(" - Costs: 30 bps one-way charged on every flip (|delta position|) x NAV. Long-only -> no borrow.")
print(" - Returns: PRICE-ONLY (BTC pays no yield); same basis for timing and buy-hold.")
print(" - Survivorship: BTC is the single SURVIVING crypto that 1000x'd -- the sample is")
print("   itself a survivor; the hashrate co-trend is conditioned on that survival. NAMED on Signal axis.")


Honesty checklist:
 - Execution lag: signal known at month-end t, position held for month t+1 (1-month lag).
 - Costs: 30 bps one-way charged on every flip (|delta position|) x NAV. Long-only -> no borrow.
 - Returns: PRICE-ONLY (BTC pays no yield); same basis for timing and buy-hold.
 - Survivorship: BTC is the single SURVIVING crypto that 1000x'd -- the sample is
   itself a survivor; the hashrate co-trend is conditioned on that survival. NAMED on Signal axis.


## Verdict

In [9]:
print("=== Study 292 -- Bitcoin-Hashrate ===")
print()
print(f"Signal: NONE")
print(f"  Hashrate growth does not predict next-month BTC returns: HAC t = {R['reg1_t']:+.2f} (1mo),")
print(f"  {R['reg3_t']:+.2f} (3mo) -- none clears t>=2. In a horse race vs price momentum the")
print(f"  hashrate slope collapses to t = {R['horse_hash_t']:+.2f}. Co-movement is a shared up-trend.")
print()
print(f"Tradability: MIRAGE")
print(f"  The 'long when hashrate rises' rule earns {R['timing_ann']:+.0f}%/yr only by being long")
print(f"  a 1000x asset {R['tim_share']:.0%} of the time -- and still LOSES to buy-and-hold")
print(f"  (SR {R['timing_sr']:.2f} vs {R['bh_sr']:.2f}). No edge survives the only honest benchmark.")
print()
print("Survivorship: NAMED -- BTC is the surviving moonshot; the co-trend is conditioned on it.")
print()
print("Bottom line: None/Mirage -- a folk indicator that is pure trend-following")
print("on a single survivor, with zero incremental predictive content.")


=== Study 292 -- Bitcoin-Hashrate ===

Signal: NONE
  Hashrate growth does not predict next-month BTC returns: HAC t = -0.05 (1mo),
  +1.52 (3mo) -- none clears t>=2. In a horse race vs price momentum the
  hashrate slope collapses to t = -0.07. Co-movement is a shared up-trend.

Tradability: MIRAGE
  The 'long when hashrate rises' rule earns +83%/yr only by being long
  a 1000x asset 86% of the time -- and still LOSES to buy-and-hold
  (SR 0.94 vs 1.04). No edge survives the only honest benchmark.

Survivorship: NAMED -- BTC is the surviving moonshot; the co-trend is conditioned on it.

Bottom line: None/Mirage -- a folk indicator that is pure trend-following
on a single survivor, with zero incremental predictive content.
